# Critical Experiments Dashboard

Auto-scans `experiments/tests/critical/euler/` and summarizes all experiment configurations and results.

**Usage:**
1. Run all cells to build the summary tables
2. Reference experiments by their DataFrame index
3. Use the cross-experiment comparison cells at the bottom

**Auto-detected data per experiment:**
- Config (YAML): attention types, HSIC settings, masks, dataset, training params
- Metrics (`kfold_summary.json`): test/val loss, HSIC, R², sparsity
- DAG metrics (`dag_metrics.json`): soft Hamming, MEC distance, skeleton recall/precision

In [1]:
import os
import json
import glob
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from os.path import join, exists, isdir

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# Path to the critical experiments folder
EULER_DIR = join('..', 'experiments', 'tests', 'critical', 'euler')
print(f'Scanning: {os.path.abspath(EULER_DIR)}')

Scanning: c:\Users\ScipioneFrancesco\Documents\Projects\causaliT\experiments\tests\critical\euler


---
## 1. Auto-Scan & Config Summary

Reads all experiment folders and extracts the key configuration parameters into a single table.

In [3]:
def find_config(exp_path):
    """Find the config YAML file in an experiment folder."""
    candidates = glob.glob(join(exp_path, 'config*.yaml'))
    if candidates:
        return candidates[0]
    return None


def safe_get(d, *keys, default=None):
    """Safely traverse nested dicts."""
    for k in keys:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d


def extract_job_id(folder_name):
    """Extract the numeric job ID from the folder name suffix."""
    parts = folder_name.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit():
        return parts[1]
    return None


def load_config_info(exp_path, folder_name):
    """Load config YAML (raw, no OmegaConf interpolation) and extract key fields."""
    config_path = find_config(exp_path)
    if config_path is None:
        return None
    
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)
    
    exp = cfg.get('experiment', {})
    training = cfg.get('training', {})
    data = cfg.get('data', {})
    model = cfg.get('model', {})
    
    return {
        'exp_name': folder_name,
        'job_id': extract_job_id(folder_name),
        'dataset': safe_get(data, 'dataset', default='?'),
        'self_attention': safe_get(exp, 'dec_self_attention_type', default='?'),
        'cross_attention': safe_get(exp, 'dec_cross_attention_type', default='?'),
        'batch_size': safe_get(exp, 'batch_size'),
        'max_epochs': safe_get(exp, 'max_epochs'),
        'lr': safe_get(exp, 'lr'),
        'd_model': safe_get(exp, 'd_model_set'),
        'dec_layers': safe_get(exp, 'dec_layers'),
        'use_hard_masks': safe_get(training, 'use_hard_masks', default=False),
        'hsic_adaptive_bw': safe_get(training, 'hsic_adaptive_bandwidth', default=False),
        'hsic_kernel_source': safe_get(training, 'hsic_kernel_source', default='rbf'),
        'hsic_mode': safe_get(training, 'hsic_mode', default='biased'),
        'nhsic_epsilon': safe_get(training, 'nhsic_epsilon', default=0.01),
        'lambda_hsic_cross': safe_get(training, 'lambda_hsic_cross', default=0.0),
        'lambda_hsic_self': safe_get(training, 'lambda_hsic_self', default=0.0),
        'gradient_routing': safe_get(training, 'use_gradient_routing', default=False),
        'hsic_cross_mode': safe_get(training, 'hsic_cross_mode', default='?'),
        'k_fold': safe_get(training, 'k_fold', default=1),
        'optimizer': safe_get(training, 'optimizer', default='?'),
        'model_object': safe_get(model, 'model_object', default='?'),
    }


# --- Scan all experiment folders ---
exp_folders = sorted([
    d for d in os.listdir(EULER_DIR)
    if isdir(join(EULER_DIR, d))
])

config_records = []
for folder in exp_folders:
    info = load_config_info(join(EULER_DIR, folder), folder)
    if info is not None:
        config_records.append(info)
    else:
        print(f'  ⚠ No config found in {folder}')

df_config = pd.DataFrame(config_records)
print(f'Found {len(df_config)} experiments\n')
df_config

Found 20 experiments



,exp_name,job_id,dataset,self_attention,cross_attention,batch_size,max_epochs,lr,d_model,dec_layers,...,hsic_kernel_source,hsic_mode,nhsic_epsilon,lambda_hsic_cross,lambda_hsic_self,gradient_routing,hsic_cross_mode,k_fold,optimizer,model_object
0,test_D2_reconstruction_64157836,64157836,scm3,ToeplitzAttention,CausalCrossAttention,128,100,0.001,48,2,...,rbf,biased,0.01,0.0,0.0,False,?,1,adamw,SingleCausalLayer
1,test_D3_1_hsic_high_bs_64315144,64315144,scm3,ToeplitzAttention,CausalCrossAttention,2048,100,0.001,48,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer
2,test_D3_1_hsic_high_bs_grok_64328070,64328070,scm3,ToeplitzAttention,CausalCrossAttention,2048,1000,0.001,48,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer
3,test_D3_1_hsic_high_bs_sigma_fix_d12_64364019,64364019,scm3,ToeplitzAttention,CausalCrossAttention,2048,100,0.001,12,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer
4,test_D3_1_hsic_high_bs_sigma_fix_d24_64360318,64360318,scm3,ToeplitzAttention,CausalCrossAttention,2048,100,0.001,24,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer
5,test_D3_1_hsic_high_bs_sigma_fix_hard_d12_6436...,64363959,scm3,ToeplitzAttention,CausalCrossAttention,2048,100,0.001,12,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer
6,test_D3_1_hsic_high_bs_sigma_fix_hard_d24_6436...,64362856,scm3,ToeplitzAttention,CausalCrossAttention,2048,100,0.001,24,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer
7,test_D3_1_hsic_high_bs_sigma_fixed_64349255,64349255,scm3,ToeplitzAttention,CausalCrossAttention,2048,100,0.001,48,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer
8,test_D3_1_hsic_high_bs_sigma_fixed_hard_64329182,64329182,scm3,ToeplitzAttention,CausalCrossAttention,2048,100,0.001,48,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer
9,test_D3_2_hsic_high_bs_hard_64321366,64321366,scm3,ToeplitzAttention,CausalCrossAttention,2048,100,0.001,48,2,...,rbf,biased,0.01,1.0,1.0,True,per_variable,1,adamw,SingleCausalLayer


### Compact Config View
Only the columns that vary across experiments — helps identify what each experiment tests.

In [ ]:
# Show only columns that differ across experiments
varying_cols = [c for c in df_config.columns if df_config[c].nunique() > 1 and c != 'exp_name' and c != 'job_id']
display_cols = ['exp_name'] + varying_cols
df_config[display_cols]

---
## 2. Metrics Summary

Loads `kfold_summary.json` and `dag_metrics.json` from each experiment and merges with configs.

In [ ]:
def load_kfold_metrics(exp_path):
    """Load key metrics from kfold_summary.json."""
    path = join(exp_path, 'kfold_summary.json')
    if not exists(path):
        return {}
    with open(path, 'r') as f:
        summary = json.load(f)
    
    stats = summary.get('statistics', {})
    result = {}
    # Extract mean values from statistics
    for key in ['test_loss_x', 'test_x_r2', 'test_x_rmse', 'test_x_mae',
                'test_hsic_cross', 'test_hsic_self', 'test_hsic_reg',
                'val_loss_x', 'val_x_r2', 'val_hsic_cross', 'val_hsic_self', 'val_hsic_reg',
                'test_self_score_sparse', 'test_cross_score_sparse',
                'trainable_params', 'avg_time_per_epoch', 'total_training_time']:
        if key in stats:
            result[key] = stats[key].get('mean', None)
    
    result['completed_folds'] = summary.get('completed_folds', 0)
    result['total_folds'] = summary.get('total_folds', 0)
    return result


def load_dag_metrics(exp_path):
    """Load DAG metrics from eval/eval_attention_scores/files/dag_metrics.json."""
    path = join(exp_path, 'eval', 'eval_attention_scores', 'files', 'dag_metrics.json')
    if not exists(path):
        return {}
    with open(path, 'r') as f:
        dag = json.load(f)
    
    result = {}
    for key in ['soft_hamming_cross', 'soft_hamming_self', 'mec_distance']:
        if key in dag and isinstance(dag[key], dict):
            result[f'{key}_mean'] = dag[key].get('mean', None)
            result[f'{key}_best'] = dag[key].get('best', None)
    
    result['mec_membership_rate'] = dag.get('mec_membership_rate', None)
    
    # Extract per-fold MEC details from the first fold
    mec_per_fold = safe_get(dag, 'mec_distance', 'per_fold', default={})
    if mec_per_fold:
        first_fold = list(mec_per_fold.values())[0]
        if isinstance(first_fold, dict):
            result['skeleton_recall'] = first_fold.get('skeleton_recall', None)
            result['skeleton_precision'] = first_fold.get('skeleton_precision', None)
            result['in_mec'] = first_fold.get('in_mec', None)
    
    return result


# --- Load metrics for all experiments ---
metric_records = []
for folder in exp_folders:
    exp_path = join(EULER_DIR, folder)
    if not isdir(exp_path):
        continue
    kfold = load_kfold_metrics(exp_path)
    dag = load_dag_metrics(exp_path)
    record = {'exp_name': folder, **kfold, **dag}
    metric_records.append(record)

df_metrics = pd.DataFrame(metric_records)

# Merge config + metrics
df = df_config.merge(df_metrics, on='exp_name', how='left')
print(f'Unified table: {len(df)} experiments, {len(df.columns)} columns')
df

### Key Metrics View

In [ ]:
key_cols = [
    'exp_name', 'dataset', 'use_hard_masks', 'hsic_adaptive_bw', 'hsic_kernel_source',
    'max_epochs',
    'test_x_r2', 'test_hsic_cross', 'test_hsic_self',
    'soft_hamming_self_mean', 'soft_hamming_cross_mean',
    'mec_distance_mean', 'skeleton_recall', 'mec_membership_rate'
]
available = [c for c in key_cols if c in df.columns]
df[available].round(4)

---
## 3. Experiment Groups

Pre-defined groups matching the categories in `docs/CRITICAL_EXP.md`.

In [ ]:
# --- Define experiment groups ---
# You can reference experiments by index or by name

GROUPS = {
    'bandwidth_adaptive_vs_fixed': {
        'description': 'Adaptive (median) vs Fixed bandwidth, with/without hard mask',
        'experiments': [
            'test_D3_1_hsic_high_bs_64315144',            # adaptive, no mask
            'test_D3_2_hsic_high_bs_hard_64321366',       # adaptive, hard mask
            'test_D3_1_hsic_high_bs_sigma_fixed_64349255', # fixed, no mask
            'test_D3_1_hsic_high_bs_sigma_fixed_hard_64329182', # fixed, hard mask
        ]
    },
    'bandwidth_fixed_dmodel': {
        'description': 'Fixed bandwidth experiments with different d_model settings',
        'experiments': [
            'test_D3_1_hsic_high_bs_sigma_fix_d12_64364019',
            'test_D3_1_hsic_high_bs_sigma_fix_d24_64360318',
            'test_D3_1_hsic_high_bs_sigma_fix_hard_d12_64363959',
            'test_D3_1_hsic_high_bs_sigma_fix_hard_d24_64362856',
        ]
    },
    'continuous_s': {
        'description': 'Continuous S (uniform) — learned vs oracle',
        'experiments': [
            'test_D3_cont_learned_64428116',
            'test_D3_cont_learned_64429750',
            'test_D3_cont_oracle_64428104',
            'test_D3_cont_oracle_64429756',
        ]
    },
    'dirac_kernel': {
        'description': 'Dirac/RBF kernel for discrete S',
        'experiments': [
            'test_D3_disc_dirac_64431751',
            "test_D3_1_hsic_high_bs_64315144"
        ]
    },
    'early_experiments': {
        'description': 'D2-D5 early diagnostic experiments',
        'experiments': [
            'test_D2_reconstruction_64157836',
            'test_D3_hsic_per_variable_64157788',
            'test_D4_structural_64157759',
            'test_D5_joint_64157738',
        ]
    },
    "grokking_continuous" : {
        "description": "when keep minimizing the HSIC, do we get closer to the true DAG?",
        "experiments": [
            "test_D3_cont_learned_64429712"
        ]
    },
    'nhsic_vs_hsic': {
        'description': 'Normalized HSIC (Ma et al. 2020) vs standard biased HSIC — continuous S, 100 epochs',
        'experiments': [
            'test_D3_cont_nhsic_64447703',        # nHSIC: hsic_mode=normalized
            'test_D3_cont_learned_64428116',       # HSIC: hsic_mode=biased, 100ep
            'test_D3_cont_learned_64429712',       # HSIC: hsic_mode=biased, 1000ep
        ]
    },
}

for name, group in GROUPS.items():
    matched = df[df['exp_name'].isin(group['experiments'])]
    print(f"\n{'='*80}")
    print(f"Group: {name}")
    print(f"Description: {group['description']}")
    print(f"Matched: {len(matched)}/{len(group['experiments'])} experiments")
    if len(matched) > 0:
        display(matched[available].round(4))

---
## 4. Training Curve Comparison

Compare training trajectories (HSIC, loss, R²) across selected experiments.

In [ ]:
def load_training_logs(exp_path, fold='k_0'):
    """
    Load training metrics CSV from an experiment fold.
    Returns a DataFrame with epoch-level metrics (train/val/test).
    """
    csv_dir = join(exp_path, fold, 'logs', 'csv')
    if not exists(csv_dir):
        return None
    
    # Find the version folder
    versions = sorted(os.listdir(csv_dir))
    if not versions:
        return None
    
    metrics_path = join(csv_dir, versions[-1], 'metrics.csv')  # latest version
    if not exists(metrics_path):
        return None
    
    df_log = pd.read_csv(metrics_path)
    return df_log


def get_epoch_summary(df_log):
    """
    Aggregate per-epoch metrics from the raw log (which has multiple rows per epoch
    for train/val/test splits). Groups by epoch and takes the first non-null value.
    """
    if df_log is None or len(df_log) == 0:
        return None
    return df_log.groupby('epoch').first().reset_index()

In [ ]:
def plot_metric_comparison(exp_names, metric, title=None, fold='k_0', ax=None):
    """
    Plot a single metric across multiple experiments.
    
    Args:
        exp_names: list of experiment folder names or DataFrame indices
        metric: column name from training logs (e.g. 'val_hsic_reg', 'train_loss_x')
        title: plot title (auto-generated if None)
        fold: which k-fold to use
        ax: matplotlib axis (creates new figure if None)
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))
    
    # Resolve indices to names if needed
    resolved_names = []
    for name in exp_names:
        if isinstance(name, int):
            resolved_names.append(df.iloc[name]['exp_name'])
        else:
            resolved_names.append(name)
    
    for name in resolved_names:
        exp_path = join(EULER_DIR, name)
        logs = load_training_logs(exp_path, fold)
        epoch_df = get_epoch_summary(logs)
        if epoch_df is not None and metric in epoch_df.columns:
            label = name.replace('test_', '').replace('_hsic_high_bs', '')
            ax.plot(epoch_df['epoch'], epoch_df[metric], label=label, alpha=0.8)
        else:
            print(f'  ⚠ Metric "{metric}" not found in {name}')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel(metric)
    ax.set_title(title or f'{metric} comparison')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    return ax

### 4a. HSIC Trajectory: Continuous Learned vs Oracle

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cont_exps = GROUPS['continuous_s']['experiments']

plot_metric_comparison(cont_exps, 'val_hsic_reg', 'HSIC (reg) — Continuous S', ax=axes[0])
plot_metric_comparison(cont_exps, 'val_hsic_cross', 'HSIC cross — Continuous S', ax=axes[1])
plot_metric_comparison(cont_exps, 'val_loss_x', 'Reconstruction Loss — Continuous S', ax=axes[2])

plt.tight_layout()
plt.show()

### 4b. HSIC Trajectory: Adaptive vs Fixed Bandwidth

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bw_exps = GROUPS['bandwidth_adaptive_vs_fixed']['experiments']

plot_metric_comparison(bw_exps, 'val_hsic_reg', 'HSIC (reg) — Bandwidth Comparison', ax=axes[0])
plot_metric_comparison(bw_exps, 'val_hsic_cross', 'HSIC cross — Bandwidth Comparison', ax=axes[1])
plot_metric_comparison(bw_exps, 'val_loss_x', 'Reconstruction Loss — Bandwidth Comparison', ax=axes[2])

plt.tight_layout()
plt.show()

### Discrete, Dirac vs RBF

In [ ]:
bw_exps = GROUPS['dirac_kernel']['experiments']

plot_metric_comparison(bw_exps, 'val_hsic_reg', 'HSIC (reg) — Bandwidth Comparison')

### 4c. Custom Comparison
Use experiment indices from the DataFrame above, or experiment names.

In [ ]:
# === EDIT HERE: select experiments by index or name ===
selected = [0, 1]  # Change these to the indices you want to compare
metric = 'val_hsic_reg'  # Change to any metric from the training logs

plot_metric_comparison(selected, metric)
plt.show()

---
## 5. Multi-Metric Dashboard

Side-by-side view of multiple metrics for a selected group of experiments.

In [ ]:
def plot_dashboard(exp_names, metrics=None, fold='k_0', suptitle=None):
    """
    Plot a dashboard of multiple metrics for selected experiments.
    
    Args:
        exp_names: list of experiment names or indices
        metrics: list of metric names to plot (default: common set)
        fold: which fold to use
        suptitle: overall title
    """
    if metrics is None:
        metrics = [
            'val_loss_x', 'val_hsic_reg', 'val_hsic_cross', 'val_hsic_self',
            'train_loss_x', 'train_hsic_reg',
        ]
    
    n = len(metrics)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
    axes = np.array(axes).flatten() if n > 1 else [axes]
    
    for i, metric in enumerate(metrics):
        plot_metric_comparison(exp_names, metric, title=metric, fold=fold, ax=axes[i])
    
    # Hide unused axes
    for j in range(n, len(axes)):
        axes[j].set_visible(False)
    
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Dashboard for continuous S experiments
plot_dashboard(
    GROUPS['continuous_s']['experiments'],
    suptitle='Continuous S: Learned vs Oracle'
)

---
## 6. Quick Reference: Experiment Lookup

Use this cell to quickly inspect a single experiment's full config and metrics.

In [ ]:
def inspect_experiment(idx_or_name):
    """
    Print detailed info for a single experiment.
    
    Args:
        idx_or_name: DataFrame index (int) or experiment name (str)
    """
    if isinstance(idx_or_name, int):
        row = df.iloc[idx_or_name]
    else:
        matches = df[df['exp_name'] == idx_or_name]
        if len(matches) == 0:
            print(f'Experiment "{idx_or_name}" not found')
            return
        row = matches.iloc[0]
    
    print(f"{'='*60}")
    print(f"Experiment: {row['exp_name']}")
    print(f"{'='*60}")
    
    print(f"\n--- Configuration ---")
    config_cols = ['dataset', 'self_attention', 'cross_attention', 'batch_size',
                   'max_epochs', 'lr', 'd_model', 'dec_layers', 'use_hard_masks',
                   'hsic_adaptive_bw', 'hsic_kernel_source', 'lambda_hsic_cross',
                   'lambda_hsic_self', 'gradient_routing', 'hsic_cross_mode']
    for col in config_cols:
        if col in row.index:
            print(f"  {col:30s}: {row[col]}")
    
    print(f"\n--- Test Metrics ---")
    metric_cols = [c for c in row.index if c.startswith('test_') or c.startswith('soft_') 
                   or c.startswith('mec_') or c.startswith('skeleton_')]
    for col in metric_cols:
        val = row[col]
        if pd.notna(val):
            if isinstance(val, float):
                print(f"  {col:30s}: {val:.6f}")
            else:
                print(f"  {col:30s}: {val}")


# === EDIT HERE: change the index ===
inspect_experiment(0)

---
## 7. Cross-Experiment Evaluation Template

Use these cells as templates for custom cross-experiment analyses.

In [ ]:
# Example: Compare hard mask vs learned for the same dataset
if 'use_hard_masks' in df.columns and 'test_hsic_reg' in df.columns:
    comparison = df.groupby(['dataset', 'use_hard_masks']).agg({
        'test_hsic_reg': ['mean', 'std', 'count'],
        'test_x_r2': ['mean'],
    }).round(6)
    display(comparison)

In [ ]:
# Example: Scatter plot of HSIC vs R² across all experiments
if 'test_hsic_reg' in df.columns and 'test_x_r2' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    scatter_df = df.dropna(subset=['test_hsic_reg', 'test_x_r2'])
    
    if len(scatter_df) > 0:
        colors = scatter_df['use_hard_masks'].map({True: 'red', False: 'blue'})
        ax.scatter(scatter_df['test_hsic_reg'], scatter_df['test_x_r2'], 
                   c=colors, s=80, alpha=0.7, edgecolors='black')
        
        # Annotate each point with its index
        for idx, row in scatter_df.iterrows():
            ax.annotate(str(idx), (row['test_hsic_reg'], row['test_x_r2']),
                        fontsize=8, ha='center', va='bottom')
        
        ax.set_xlabel('Test HSIC (reg)')
        ax.set_ylabel('Test R²')
        ax.set_title('HSIC vs R² (red=hard mask, blue=learned)')
        ax.grid(True, alpha=0.3)
    plt.show()